# Penguins of Kyoto - Social Network Analysis

## Processing

### Import libraries

In [ ]:
# Standard library
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict

# Data manipulation & numerics
import pandas as pd
import numpy as np
import scipy as sp

# Network analysis
import networkx as nx
import community.community_louvain as community  # Louvain method

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

### Import data

In [ ]:
# Folder where GEXF files are stored
DATA_DIR = "data/gexf"

# List of GEXF network files
gexf_files = [
    "Penguin_Complicated.gexf",
    "Penguin_Couples.gexf",
    "Penguin_Enemies.gexf",
    "Penguin_Exes.gexf",
    "Penguin_Family.gexf",
    "Penguin_Friends.gexf",
]

# Final multigraph (multiple relations can exist between the same pair)
G = nx.MultiDiGraph()

# Load each GEXF file and merge it into the final graph
for file in gexf_files:
    path = os.path.join(DATA_DIR, file)
    relationship = file.split("_")[1].split(".")[0]  # extract relationship type

    print(f"→ Loading {relationship} network...")
    G_read = nx.read_gexf(path)

    # Add edges and store their relationship type
    for u, v in G_read.edges():
        G.add_edge(u, v, relationship_type=relationship)

# Clean node names (remove trailing spaces)
mapping = {node: node.strip() for node in G.nodes()}
G = nx.relabel_nodes(G, mapping)

# Load metadata (gender, name, info)
metadata_path = "data/csv/Penguins of Kyoto - Metadata.csv"
meta = pd.read_csv(metadata_path)

# Clean Name field
meta["Name"] = meta["Name"].astype(str).str.strip()

# Map node → color based on gender
gender_map = {}
for _, row in meta.iterrows():
    name = str(row["Name"]).strip()
    gender = str(row["Gender"]).strip().lower()

    if gender.startswith("m"):
        gender_map[name] = "skyblue"      # male
    elif gender.startswith("f"):
        gender_map[name] = "lightpink"    # female
    else:
        gender_map[name] = "gray"         # unknown gender


# Assegno il gender al grafo
gender_attr_map = dict(zip(meta["Name"], meta["Gender"]))
nx.set_node_attributes(G, gender_attr_map, name="gender")

# Checks that all nodes have the gender attribute
missing = [n for n in G.nodes() if G.nodes[n].get("gender") is None]
print("\nNodes without gender attribute:", missing)

print(f"\nFinal network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")


## Structure

### Draw network graph

In [ ]:
# Compute node positions (force-directed layout)
pos = nx.spring_layout(G, seed=42, k=0.8)
#pos = nx.arf_layout(G, max_iter=50)


# --- Force-directed (consigliati) ---
#pos = nx.spring_layout(G, seed=42, k=1.2, iterations=200)
#pos = nx.kamada_kawai_layout(G, weight=None)
#pos = nx.forceatlas2_layout(G, seed=42, max_iter=300, scaling_ratio=2.0)
#pos = nx.arf_layout(G, max_iter=200)

# --- Geometrici ---
#pos = nx.circular_layout(G, scale=2)
#pos = nx.shell_layout(G, scale=2)
#pos = nx.spiral_layout(G, scale=2)

# --- Strutturali ---
#pos = nx.spectral_layout(G, scale=2)
#pos = nx.planar_layout(G, scale=2)

# --- Basati su traversa ---
#pos = nx.bfs_layout(G, start=max(G.degree, key=lambda x: x[1])[0])

# --- Random ---
#pos = nx.random_layout(G, seed=42)

# --- Multipartite (serve attributo nei nodi) ---
#pos = nx.multipartite_layout(G, subset_key="gender", align="vertical", scale=2)

# --- Bipartite (serve divisione nodi) ---
#pos = nx.bipartite_layout(G, nodes=list(G.nodes())[:len(G)//2], scale=2)

#pos = nx.rescale_layout_dict(pos, scale=2)


# Color assigned to each relationship type
color_map = {
    "Couples": "red",
    "Exes": "blue",
    "Complicated": "purple",
    "Family": "lightgray",
    "Friends": "green",
    "Enemies": "darkorange"
}

# Edge colors based on relationship type
edge_colors = [
    color_map[G[u][v][k]['relationship_type']]
    for u, v, k in G.edges(keys=True)
]

# Node colors based on gender
node_colors = [gender_map.get(node, "gray") for node in G.nodes()]


# Create figure
plt.figure(figsize=(16, 16))
plt.title("Penguins of Kyoto Relationship Network", fontsize=20)
plt.axis('off')

# Legend: gender
plt.scatter([], [], c="skyblue", label="Male", s=150, edgecolors="black")
plt.scatter([], [], c="lightpink", label="Female", s=150, edgecolors="black")

# Legend: relation types
for label, color in color_map.items():
    plt.plot([], [], color=color, label=label, linewidth=4)

plt.legend(
    title="Relationship Types",
    loc="upper right",
    fontsize=8,
    title_fontsize=10,
    frameon=True,
    facecolor="white",
    edgecolor="black",
    labelspacing=1.2
)

# Draw nodes
nx.draw_networkx_nodes(
    G, pos,
    node_color=node_colors,
    node_size=500,
    alpha=0.8
)

# Draw directed edges
nx.draw_networkx_edges(
    G, pos,
    edge_color=edge_colors,
    arrows=True,
    arrowstyle='-|>',
    arrowsize=10,
    alpha=0.6
)

# Draw node labels
nx.draw_networkx_labels(G, pos, font_size=9, font_color='black')

plt.show()


### Basic info

In [ ]:
# Basic graph size
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

# Check if graph is directed
is_direct_result = G.is_directed()
print("Graph is directed:", is_direct_result)

# Check if graph is bipartite
is_bipartite_result = nx.is_bipartite(G)
print("Graph is bipartite:", is_bipartite_result)

# Check if graph is unweighted
is_unweighted_result = all('weight' not in data for _, _, data in G.edges(data=True))
print("Graph is unweighted:", is_unweighted_result)

# Compute weakly connected components
components = list(nx.weakly_connected_components(G))
print("Number of connected components:", len(components))



# Convert to undirected version
G_und = G.to_undirected()

# Compute average shortest path length
average_path_length = nx.average_shortest_path_length(G_und)

# Compute graph diameter
diameter = nx.diameter(G_und)

# Compute network density (ratio of actual edges vs possible edges)
density = nx.density(G)


print(f"Average path length (undirected): {average_path_length:.2f}")
print(f"Diameter (undirected): {diameter}")
print(f"Network density: {density}")


### Node degrees

In [ ]:
# Compute degree of each node (number of connections)
degree_dict = dict(G.degree())

# Sort nodes by degree (highest → lowest)
sorted_degrees = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)

# Print ranked nodes by degree
print("Nodes sorted by degree (highest to lowest):")
for node, degree in sorted_degrees:
    print(f"Node: {node}, Degree: {degree}")

# Compute average degree
average_degree = sum(degree_dict.values()) / G.number_of_nodes()
print(f"Average degree of the network: {average_degree:.2f}")


### Degree graph

In [ ]:
# Assign a color to each node based on its degree
colors = [degree_dict[node] for node in G.nodes()]

plt.figure(figsize=(12, 12))

# Compute node layout
pos = nx.spring_layout(G, seed=42, k=0.8)

# Draw nodes colored by degree
nodes = nx.draw_networkx_nodes(
    G, pos,
    node_size=500,
    node_color=colors,
    cmap=plt.cm.cool
)

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color="gray")

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_color="black")

# Color bar
plt.colorbar(nodes, label='Node degree')

plt.title("Penguins of Kyoto network - Degree of penguins connection")
plt.show()


### In-out degree graph

In [ ]:
# Compute indegree and outdegree for each node
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())

# Top 6 nodes by indegree
top_in_degrees = sorted(in_degrees.items(), key=lambda x: x[1], reverse=True)[:6]
print("Top 6 nodes by indegree:")
for node, indeg in top_in_degrees:
    print(f"Node: {node}, Indegree: {indeg}")

# Top 6 nodes by outdegree
top_out_degrees = sorted(out_degrees.items(), key=lambda x: x[1], reverse=True)[:6]
print("Top 6 nodes by outdegree:")
for node, outdeg in top_out_degrees:
    print(f"Node: {node}, Outdegree: {outdeg}")

# --- Plot top 6 indegree and outdegree ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

# ---------- INDEGREE ----------
top_in_nodes = [node for node, _ in top_in_degrees]
subgraph_in = G.subgraph(top_in_nodes)
pos_in = nx.spring_layout(subgraph_in)

# Global color range for indegree
global_in_min = min(in_degrees.values()) if in_degrees else 0
global_in_max = max(in_degrees.values()) if in_degrees else 1
vmin_in = 1 if global_in_min < 1 else global_in_min

# Draw nodes
nodes_in = nx.draw_networkx_nodes(
    subgraph_in, pos_in,
    node_size=400,
    node_color=[in_degrees[node] for node in subgraph_in.nodes()],
    cmap=plt.cm.cool,
    vmin=vmin_in, vmax=global_in_max,
    ax=ax1
)

# Move labels slightly up
label_offset_sub = 0.08
pos_in_labels = {n: (pos_in[n][0], pos_in[n][1] + label_offset_sub) for n in subgraph_in.nodes()}

# Draw labels only
nx.draw_networkx_labels(subgraph_in, pos_in_labels, font_size=12, font_color="black", ax=ax1)

ax1.set_title("Top 6 Penguins by Indegree", fontsize=16)

# ---------- OUTDEGREE ----------
top_out_nodes = [node for node, _ in top_out_degrees]
subgraph_out = G.subgraph(top_out_nodes)
pos_out = nx.spring_layout(subgraph_out)

# Global color range for outdegree
global_out_min = min(out_degrees.values()) if out_degrees else 0
global_out_max = max(out_degrees.values()) if out_degrees else 1
vmin_out = 1 if global_out_min < 1 else global_out_min

# Draw nodes
nodes_out = nx.draw_networkx_nodes(
    subgraph_out, pos_out,
    node_size=400,
    node_color=[out_degrees[node] for node in subgraph_out.nodes()],
    cmap=plt.cm.cool,
    vmin=vmin_out, vmax=global_out_max,
    ax=ax2
)

# Label offset
pos_out_labels = {n: (pos_out[n][0], pos_out[n][1] + label_offset_sub) for n in subgraph_out.nodes()}

# Draw labels only
nx.draw_networkx_labels(subgraph_out, pos_out_labels, font_size=12, font_color="black", ax=ax2)

ax2.set_title("Top 6 Penguins by Outdegree", fontsize=16)

# Colorbars
fig.colorbar(nodes_in, ax=ax1, label='Indegree')
fig.colorbar(nodes_out, ax=ax2, label='Outdegree')

# ---------- Adjust axis limits to include labels ----------

# Indegree axis limits
xs_in = [p[0] for p in pos_in.values()]
ys_in = [p[1] for p in pos_in.values()]
lxs_in = [p[0] for p in pos_in_labels.values()]
lys_in = [p[1] for p in pos_in_labels.values()]
xmin_i, xmax_i = min(xs_in), max(xs_in)
ymin_i, ymax_i = min(ys_in), max(ys_in)
xmin_i_final = min(xmin_i, min(lxs_in))
xmax_i_final = max(xmax_i, max(lxs_in))
ymin_i_final = min(ymin_i, min(lys_in))
ymax_i_final = max(ymax_i, max(lys_in))
xpad_i = 0.06 * (xmax_i_final - xmin_i_final) if xmax_i_final > xmin_i_final else 0.06
ypad_i = 0.08 * (ymax_i_final - ymin_i_final) if ymax_i_final > ymin_i_final else 0.08
ax1.set_xlim(xmin_i_final - xpad_i, xmax_i_final + xpad_i)
ax1.set_ylim(ymin_i_final - ypad_i, ymax_i_final + ypad_i)

# Outdegree axis limits
xs_o = [p[0] for p in pos_out.values()]
ys_o = [p[1] for p in pos_out.values()]
lxs_o = [p[0] for p in pos_out_labels.values()]
lys_o = [p[1] for p in pos_out_labels.values()]
xmin_o, xmax_o = min(xs_o), max(xs_o)
ymin_o, ymax_o = min(ys_o), max(ys_o)
xmin_o_final = min(xmin_o, min(lxs_o))
xmax_o_final = max(xmax_o, max(lxs_o))
ymin_o_final = min(ymin_o, min(lys_o))
ymax_o_final = max(ymax_o, max(lys_o))
xpad_o = 0.06 * (xmax_o_final - xmin_o_final) if xmax_o_final > xmin_o_final else 0.06
ypad_o = 0.08 * (ymax_o_final - ymin_o_final) if ymax_o_final > ymin_o_final else 0.08
ax2.set_xlim(xmin_o_final - xpad_o, xmax_o_final + xpad_o)
ax2.set_ylim(ymin_o_final - ypad_o, ymax_o_final + ypad_o)

plt.show()


## Centrality

### Degree centrality

In [ ]:
# Compute degree centrality for each node
degree_centrality = nx.degree_centrality(G)

# Sort nodes by centrality (highest → lowest)
sorted_degree_centrality = {
    k: v for k, v in sorted(degree_centrality.items(), key=lambda item: item[1], reverse=True)
}

# Print results
print("Degree centrality (highest to lowest):")
for node, centrality in sorted_degree_centrality.items():
    print(f"Node {node}: {centrality:.4f}")


### Betweenness centrality

In [ ]:
# Compute betweenness centrality for each node
betweenness_centrality = nx.betweenness_centrality(G)

# Sort nodes by betweenness (highest → lowest)
sorted_betweenness_centrality = {
    k: v for k, v in sorted(betweenness_centrality.items(), key=lambda item: item[1], reverse=True)
}

# Print results
print("Betweenness centrality (highest to lowest):")
for node, centrality in sorted_betweenness_centrality.items():
    print(f"Node {node}: {centrality:.4f}")


### Closeness centrality

In [ ]:
# Compute closeness centrality for each node
closeness_centrality = nx.closeness_centrality(G)

# Sort nodes by closeness (highest → lowest)
sorted_closeness = sorted(closeness_centrality.items(), key=lambda item: item[1], reverse=True)

# Print results
print("Closeness centrality (highest to lowest):")
for node, closeness in sorted_closeness:
    print(f"{node}: {closeness:.4f}")


### Load centrality

In [ ]:
# Compute load centrality for each node
load_centrality = nx.load_centrality(G)

# Sort nodes by load centrality (highest → lowest)
sorted_load_centrality = sorted(load_centrality.items(), key=lambda item: item[1], reverse=True)

# Print results
print("Load centrality (highest to lowest):")
for node, load in sorted_load_centrality:
    print(f"{node}: {load:.4f}")


### Eigenvector centrality

In [ ]:
# ---- Eigenvector centrality (UNDIRECTED) ----

# Convert MultiDiGraph → simple undirected graph
G_simple = nx.Graph(G)

# Compute eigenvector centrality
eigen_centrality = nx.eigenvector_centrality(G_simple, max_iter=1000)

# Sort results (highest → lowest)
sorted_eigen = sorted(eigen_centrality.items(), key=lambda x: x[1], reverse=True)

print("\nEigenvector Centrality (UNDIRECTED) (top → bottom):")
for node, value in sorted_eigen:
    print(f"{node}: {value:.4f}")


# ---- Eigenvector centrality (DIRECTED) ----

# Convert MultiDiGraph → directed simple graph
G_directed = nx.DiGraph(G)

# Compute directed eigenvector centrality
eigen_directed = nx.eigenvector_centrality(G_directed, max_iter=2000)

# Sort results (highest → lowest)
sorted_eigen_directed = sorted(eigen_directed.items(), key=lambda x: x[1], reverse=True)

print("\nEigenvector Centrality (DIRECTED) (top → bottom):")
for node, value in sorted_eigen_directed:
    print(f"{node}: {value:.4f}")

### Top nodes by centrality type

In [ ]:
# Top 5 nodes by degree centrality
top_degree = sorted(degree_dict.items(), key=lambda item: item[1], reverse=True)[:5]
print("Top 5 nodes by degree:", top_degree)

# Top 5 nodes by betweenness centrality
top_betweenness = sorted(betweenness_centrality.items(), key=lambda item: item[1], reverse=True)[:5]
print("Top 5 nodes by betweenness:", top_betweenness)

# Top 5 nodes by closeness centrality
top_closeness = sorted(closeness_centrality.items(), key=lambda item: item[1], reverse=True)[:5]
print("Top 5 nodes by closeness:", top_closeness)

# Top 5 nodes by load centrality
top_load = sorted(load_centrality.items(), key=lambda item: item[1], reverse=True)[:5]
print("Top 5 nodes by load:", top_load)


## Community

In [ ]:


# ---- Relationship weights (interaction strength) ----
RELATION_WEIGHTS = {
    "Friends": 1.0,
    "Couples": 1.0,
    "Complicated": 0.7,
    "Exes": 0.5,
    "Enemies": 0.0,
    "Family": 0.3
}

# ---- Convert MultiDiGraph → simple weighted undirected graph ----
weighted_graph = nx.Graph()

for u, v, data in G.edges(data=True):
    rel_type = data.get("relationship_type", "")
    weight = RELATION_WEIGHTS.get(rel_type, 0.1)

    if weighted_graph.has_edge(u, v):
        weighted_graph[u][v]["weight"] += weight  # accumulate weight
    else:
        weighted_graph.add_edge(u, v, weight=weight)

print(f"Weighted graph: {weighted_graph.number_of_nodes()} nodes, {weighted_graph.number_of_edges()} edges")

# ---- Weighted clustering and transitivity ----
local_clustering = nx.clustering(weighted_graph, weight='weight')
avg_clustering = nx.average_clustering(weighted_graph, weight='weight')
global_transitivity = nx.transitivity(weighted_graph)


# ---- Weighted cliquishness (neighbors only) ----
def weighted_cliquishness(G, node):
    neighbors = list(G.neighbors(node))
    if len(neighbors) < 2:
        return 0.0

    subgraph = G.subgraph(neighbors)  # neighbor-only subgraph

    # Weighted density
    possible_edges = len(neighbors)*(len(neighbors)-1)/2
    actual_weight = sum(d.get("weight", 1) for _, _, d in subgraph.edges(data=True))
    max_possible_weight = possible_edges * max(RELATION_WEIGHTS.values())
    density_score = actual_weight / max_possible_weight

    # Neighbor strength adjustment
    neighbor_strengths = [G.degree(n, weight="weight") for n in neighbors]
    strength_factor = np.mean(neighbor_strengths) / max(dict(G.degree(weight="weight")).values())

    return density_score * strength_factor


# ---- Extended cliquishness (ego-radius = 2) ----
def extended_ego_cliquishness(G, node):
    ego_graph = nx.ego_graph(G, node, radius=2)
    n_nodes = ego_graph.number_of_nodes()
    if n_nodes < 3:
        return 0.0

    possible_edges = n_nodes*(n_nodes-1)/2
    actual_weight = sum(d.get("weight", 1) for _, _, d in ego_graph.edges(data=True))
    max_possible_weight = possible_edges * max(RELATION_WEIGHTS.values())

    return actual_weight / max_possible_weight


# ---- Degree-normalized cliquishness ----
def degree_normalized_cliquishness(G, node):
    c = nx.clustering(G, node, weight="weight")
    k = G.degree(node, weight="weight")
    if k == 0:
        return 0.0
    return c / (1 + np.log1p(k))


# ---- Build dataframe of metrics ----
weighted_degrees = dict(weighted_graph.degree(weight="weight"))
degree_centrality = nx.degree_centrality(weighted_graph)

metrics_df = pd.DataFrame({
    "node": list(weighted_graph.nodes()),
    "weighted_degree": [weighted_degrees[n] for n in weighted_graph.nodes()],
    "degree_centrality": [degree_centrality[n] for n in weighted_graph.nodes()],
    "local_clustering": [local_clustering[n] for n in weighted_graph.nodes()],
    "weighted_cliquishness": [weighted_cliquishness(weighted_graph, n) for n in weighted_graph.nodes()],
    "extended_ego_cliquishness": [extended_ego_cliquishness(weighted_graph, n) for n in weighted_graph.nodes()],
    "degree_normalized_cliquishness": [degree_normalized_cliquishness(weighted_graph, n) for n in weighted_graph.nodes()]
}).set_index("node")

# Difference between cliquishness and clustering
metrics_df["cliquishness_vs_clustering_delta"] = (
    metrics_df["weighted_cliquishness"] - metrics_df["local_clustering"]
)

# ---- Global results ----
print(f"Average weighted clustering: {avg_clustering:.4f}")
print(f"Global transitivity: {global_transitivity:.4f}")
print(f"Mean weighted cliquishness: {metrics_df['weighted_cliquishness'].mean():.4f}")
print(f"Mean extended ego cliquishness: {metrics_df['extended_ego_cliquishness'].mean():.4f}")
print(f"Mean degree-normalized cliquishness: {metrics_df['degree_normalized_cliquishness'].mean():.4f}")

# ---- Node rankings ----
print("\nTop nodes by weighted cliquishness:")
print(metrics_df.sort_values("weighted_cliquishness", ascending=False))

print("\nTop nodes by extended ego cliquishness:")
print(metrics_df.sort_values("extended_ego_cliquishness", ascending=False))

# ---- Export results ----
output_file = "export/cliquishness_clustering_results.csv"
metrics_df.to_csv(output_file)
print(f"\nResults saved to: {output_file}")


In [ ]:
# --- SCATTER PLOTS: weighted degree vs cliquishness metrics ---
fig, axes = plt.subplots(1, 3, figsize=(18,5))

axes[0].scatter(metrics_df["weighted_degree"], metrics_df["weighted_cliquishness"], alpha=0.7, color="steelblue")
axes[0].set_title("Weighted degree vs Weighted cliquishness")
axes[0].set_xlabel("Weighted degree")
axes[0].set_ylabel("Value")

axes[1].scatter(metrics_df["weighted_degree"], metrics_df["extended_ego_cliquishness"], alpha=0.7, color="darkorange")
axes[1].set_title("Weighted degree vs Extended ego cliquishness")
axes[1].set_xlabel("Weighted degree")
axes[1].set_ylabel("Value")

axes[2].scatter(metrics_df["weighted_degree"], metrics_df["degree_normalized_cliquishness"], alpha=0.7, color="seagreen")
axes[2].set_title("Weighted degree vs Degree-normalized cliquishness")
axes[2].set_xlabel("Weighted degree")
axes[2].set_ylabel("Value")

# Enable grid
for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


# --- HISTOGRAMS OF ALL METRICS (combined) ---
plt.figure(figsize=(10,6))
metrics_df[[
    "local_clustering",
    "weighted_cliquishness",
    "extended_ego_cliquishness",
    "degree_normalized_cliquishness"
]].plot.hist(bins=20, alpha=0.6, figsize=(10,6))

plt.title("Distribution of clustering and cliquishness metrics")
plt.xlabel("Metric value")
plt.ylabel("Frequency")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# --- SEPARATE HISTOGRAMS FOR EACH METRIC ---
metrics_list = [
    "local_clustering",
    "weighted_cliquishness",
    "extended_ego_cliquishness",
    "degree_normalized_cliquishness"
]

for metric in metrics_list:
    plt.figure(figsize=(8,5))
    plt.hist(metrics_df[metric], bins=20, alpha=0.7, color="skyblue", edgecolor="black")
    plt.title(f"Distribution of metric: {metric}")
    plt.xlabel("Metric value")
    plt.ylabel("Frequency")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# --- NETWORK GRAPH COLORED BY LOCAL CLUSTERING ---
fig, ax = plt.subplots(figsize=(14,12))
pos = nx.spring_layout(weighted_graph, seed=42)

# Node colors = clustering values
node_colors = [metrics_df.loc[n, "local_clustering"] for n in weighted_graph.nodes()]

nx.draw_networkx_edges(weighted_graph, pos, alpha=0.3)

nx.draw_networkx_nodes(
    weighted_graph, pos,
    node_color=node_colors,
    cmap=plt.cm.cool,
    node_size=600,
    alpha=0.9
)

nx.draw_networkx_labels(weighted_graph, pos, font_size=8)

# Colorbar
sm = plt.cm.ScalarMappable(
    cmap=plt.cm.cool,
    norm=plt.Normalize(vmin=min(node_colors), vmax=max(node_colors))
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Local clustering")

ax.set_title("Network visualization colored by local clustering", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()


# --- NETWORK GRAPH COLORED BY WEIGHTED DEGREE ---
fig, ax = plt.subplots(figsize=(14,12))
pos = nx.spring_layout(weighted_graph, seed=42)  # keep layout stable

node_colors_degree = [metrics_df.loc[n, "weighted_degree"] for n in weighted_graph.nodes()]

nx.draw_networkx_edges(weighted_graph, pos, alpha=0.3)

nx.draw_networkx_nodes(
    weighted_graph, pos,
    node_color=node_colors_degree,
    cmap=plt.cm.cool,
    node_size=600,
    alpha=0.9
)

nx.draw_networkx_labels(weighted_graph, pos, font_size=8)

# Colorbar
sm = plt.cm.ScalarMappable(
    cmap=plt.cm.cool,
    norm=plt.Normalize(vmin=min(node_colors_degree), vmax=max(node_colors_degree))
)
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Weighted degree")

ax.set_title("Network visualization colored by weighted degree (more socially active nodes)", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()


# --- CORRELATION HEATMAP ---
plt.figure(figsize=(8,6))
sns.heatmap(
    metrics_df[[
        "local_clustering",
        "weighted_cliquishness",
        "extended_ego_cliquishness",
        "degree_normalized_cliquishness"
    ]].corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)

plt.title("Correlation between clustering and cliquishness metrics")
plt.tight_layout()
plt.show()


In [ ]:
# Convert to simple undirected graph (remove direction and multi-edges)
G_undirected = nx.Graph(G)

# Global transitivity (triangle-based clustering)
transitivity = nx.transitivity(G_undirected)
print("Transitivity:", transitivity)

# Convert to simple directed graph
G_simple = nx.DiGraph(G)

# Reciprocity of directed edges
reciprocity = nx.reciprocity(G_simple)
print("Reciprocity:", reciprocity)


In [ ]:
# Compute reciprocity separately for each relationship type
reciprocity_per_relation = {}

# Build a directed graph for each relation type
for u, v, data in G.edges(data=True):
    rel = data["relationship_type"]
    reciprocity_per_relation.setdefault(rel, nx.DiGraph()).add_edge(u, v)

# Compute reciprocity for each relation
for rel, G_rel in reciprocity_per_relation.items():
    reciprocity = nx.reciprocity(G_rel)
    print(f"{rel}: {reciprocity:.3f}")


In [ ]:
# --- Build directed subgraph for each relationship type ---
reciprocity_per_relation = {}

for u, v, data in G.edges(data=True):
    rel = data["relationship_type"]
    reciprocity_per_relation.setdefault(rel, nx.DiGraph()).add_edge(u, v)

# --- Extract labels and reciprocity values for plotting ---
labels = []
values = []

for rel, G_rel in reciprocity_per_relation.items():
    reciprocity = nx.reciprocity(G_rel)
    labels.append(rel)
    values.append(reciprocity)

# --- Bar chart of reciprocity per relation ---
plt.figure(figsize=(8, 5))
plt.bar(labels, values)
plt.xlabel("Relationship Type")
plt.ylabel("Reciprocity")
plt.title("Reciprocity by Relationship Type")
plt.ylim(0, 1)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
G_undirected = nx.Graph(G)

# Dizionario: tipo_relazione → lista di archi (senza duplicati)
edges_by_rel = {}

for u, v, data in G.edges(data=True):
    rel = data.get("relationship_type", "unknown")
    edges_by_rel.setdefault(rel, set())
    edges_by_rel[rel].add(tuple(sorted((u, v))))  # rimuove duplicati direzionali

# Calcolo omofilia per ogni tipo di relazione
homophily_gender_per_relation = {}

for rel, edge_set in edges_by_rel.items():
    same = 0
    total = 0

    for u, v in edge_set:
        # Consideriamo solo archi i cui nodi hanno effettivamente il gender
        g1 = G_undirected.nodes[u].get("gender")
        g2 = G_undirected.nodes[v].get("gender")

        if g1 is not None and g2 is not None:
            total += 1
            if g1 == g2:
                same += 1

    if total == 0:
        homophily_gender_per_relation[rel] = None
    else:
        homophily_gender_per_relation[rel] = same / total

# Stampa risultati
for rel, h in homophily_gender_per_relation.items():
    print(f"{rel}: {h}")

In [ ]:
labels = list(homophily_gender_per_relation.keys())
values = [v if v is not None else 0 for v in homophily_gender_per_relation.values()]

# New style: soft grid, minimalistic
plt.figure(figsize=(9, 5))
plt.style.use('bmh')  # different style

bars = plt.bar(labels, values, color="#69b3a2", edgecolor="black")  # softer palette

# Labels
plt.xlabel("Relationship Type")
plt.ylabel("Gender Homophily")
plt.title("Gender Homophily by Relationship Type")
plt.ylim(0, 1)

plt.xticks(rotation=25)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height + 0.02,
        f"{height:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

In [ ]:
# Input/output folders
data_dir = Path("data")
output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

# Load graph depending on file type
def load_graph(path: Path):
    ext = path.suffix.lower()

    if ext == ".gml":
        return nx.read_gml(path)

    if ext in {".graphml", ".graphmlz"}:
        return nx.read_graphml(path)

    if ext in {".edgelist", ".txt"}:
        return nx.read_edgelist(path)

    if ext == ".csv":
        df = pd.read_csv(path)
        if len(df.columns) >= 2:
            return nx.from_pandas_edgelist(df, df.columns[0], df.columns[1])
        else:
            raise ValueError(f"CSV {path} must contain at least 2 columns.")

    # Fallback: try adjacency list
    try:
        return nx.read_adjlist(path)
    except Exception as e:
        raise ValueError(f"Unrecognized file format for {path}: {e}")


# Compute distance-based metrics
def analyze_graph(G, name):
    res = {"name": name}

    # Basic info
    res['n_nodes'] = G.number_of_nodes()
    res['n_edges'] = G.number_of_edges()
    res['directed'] = G.is_directed()

    # Select largest connected component
    if G.is_directed():
        try:
            comp_sets = list(nx.strongly_connected_components(G))
            largest = max(comp_sets, key=len)
            H = G.subgraph(largest).copy().to_undirected()
            res['used_component'] = 'largest_strongly_connected'
        except Exception:
            comp_sets = list(nx.weakly_connected_components(G))
            largest = max(comp_sets, key=len)
            H = G.subgraph(largest).copy().to_undirected()
            res['used_component'] = 'largest_weakly_connected'
    else:
        comp_sets = list(nx.connected_components(G))
        largest = max(comp_sets, key=len) if comp_sets else set()
        H = G.subgraph(largest).copy()
        res['used_component'] = 'largest_connected'

    res['component_nodes'] = H.number_of_nodes()
    res['component_edges'] = H.number_of_edges()

    # Too small to compute distances
    if H.number_of_nodes() <= 1:
        res['diameter'] = None
        res['average_path_length'] = None
        res['notes'] = "component too small"
        return res

    # Diameter
    try:
        res['diameter'] = nx.diameter(H)
    except Exception as e:
        res['diameter'] = None
        res['diameter_error'] = str(e)

    # Average path length
    try:
        res['average_path_length'] = nx.average_shortest_path_length(H)
    except Exception as e:
        res['average_path_length'] = None
        res['apl_error'] = str(e)

    # Path length distribution (full or sampled)
    try:
        n = H.number_of_nodes()
        max_pairs = 200000  # threshold for full computation

        if n * (n - 1) / 2 <= max_pairs:
            # Compute all-pairs shortest paths
            lengths = dict(nx.all_pairs_shortest_path_length(H))
            dists = []
            for u in lengths:
                dists.extend(lengths[u].values())
            res['path_length_hist'] = pd.Series(dists).value_counts().sort_index().to_dict()

        else:
            # Sampling for large graphs
            nodes = list(H.nodes())
            pairs = set()
            sampled = []
            samples = 20000

            for _ in range(samples):
                a, b = random.sample(nodes, 2)
                if (a, b) not in pairs:
                    pairs.add((a, b))
                    try:
                        l = nx.shortest_path_length(H, a, b)
                        sampled.append(l)
                    except nx.NetworkXNoPath:
                        pass

            res['path_length_hist_sampled'] = pd.Series(sampled).value_counts().sort_index().to_dict()

    except Exception as e:
        res['path_length_hist_error'] = str(e)

    return res


# Process all graph files
all_results = []

for fpath in sorted(data_dir.rglob("*.*")):

    # Skip irrelevant file types
    if fpath.suffix.lower() in {'.py', '.md', '.gitignore'}:
        continue

    if fpath.is_file():
        name = fpath.stem

        try:
            G = load_graph(fpath)
        except Exception as e:
            print(f"[WARN] Cannot load {fpath.name}: {e}")
            continue

        print(f"Analyzing {fpath.name}: nodes={G.number_of_nodes()} edges={G.number_of_edges()} directed={G.is_directed()}")
        r = analyze_graph(G, name)
        all_results.append(r)


# Save results
df_res = pd.DataFrame(all_results)
df_res.to_csv(output_dir / "graph_distance_measures.csv", index=False)

with open(output_dir / "graph_distance_measures.json", "w") as fh:
    json.dump(all_results, fh, indent=2)

print("Results saved in", output_dir)


# Save plots for each graph
for r in all_results:

    base = output_dir / r['name']

    # Path length histogram
    hist = r.get('path_length_hist') or r.get('path_length_hist_sampled')
    if hist:
        keys = sorted(int(k) for k in hist.keys())
        vals = [hist[str(k)] if isinstance(list(hist.keys())[0], str) else hist[k] for k in keys]

        plt.figure()
        plt.bar(keys, vals)
        plt.xlabel("Shortest path length")
        plt.ylabel("Count")
        plt.title(f"Path length distribution: {r['name']}")
        plt.tight_layout()
        plt.savefig(base.with_suffix(".path_hist.png"))
        plt.close()

    # Degree distribution (if GML exists)
    try:
        Gpath = data_dir / (r['name'] + ".gml")

        if Gpath.exists():
            Gtmp = load_graph(Gpath)
            degs = [d for n, d in Gtmp.degree()]

            plt.figure()
            plt.hist(degs, bins=30)
            plt.xlabel("Degree")
            plt.ylabel("Count")
            plt.title(f"Degree distribution: {r['name']}")
            plt.tight_layout()
            plt.savefig(base.with_suffix(".degree_hist.png"))
            plt.close()
    except Exception:
        pass


# Display summary
display(df_res[['name', 'n_nodes', 'n_edges', 'component_nodes', 'component_edges', 'diameter', 'average_path_length']])


In [ ]:


# Load metadata (gender info)
metadata_path = "data/csv/Penguins of Kyoto - Metadata.csv"
meta = pd.read_csv(metadata_path)
meta["Name"] = meta["Name"].astype(str).str.strip()

gender_map = {}
for _, row in meta.iterrows():
    name = str(row["Name"]).strip()
    g = str(row["Gender"]).lower().strip()
    if g.startswith("m"):
        gender_map[name] = "skyblue"     # male
    elif g.startswith("f"):
        gender_map[name] = "lightpink"   # female
    else:
        gender_map[name] = "gray"        # unknown


# Load multilayer GEXF networks
DATA_DIR = "data/gexf"
gexf_files = [
    "Penguin_Complicated.gexf",
    "Penguin_Couples.gexf",
    "Penguin_Enemies.gexf",
    "Penguin_Exes.gexf",
    "Penguin_Family.gexf",
    "Penguin_Friends.gexf",
]

color_map = {
    "Couples": "red",
    "Exes": "blue",
    "Complicated": "purple",
    "Family": "orange",
    "Friends": "green",
    "Enemies": "gray"
}

G = nx.MultiGraph()

# Merge all GEXF files
for file in gexf_files:
    path = os.path.join(DATA_DIR, file)
    relation = file.split("_")[1].split(".")[0]

    print(f"→ Loading {relation} network from {path}")
    Gi = nx.read_gexf(path)

    # Add edges with relationship attribute
    for u, v in Gi.edges():
        G.add_edge(u.strip(), v.strip(), relationship_type=relation)

print(f"MultiGraph combined: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}")


# Convert to weighted simple graph
RELATION_WEIGHTS = {
    "Friends": 1.0,
    "Couples": 1.0,
    "Complicated": 0.7,
    "Exes": 0.5,
    "Enemies": 0.0,
    "Family": 0.3
}

G_simple = nx.Graph()

# Sum weights when multiple relations exist
for u, v, data in G.edges(data=True):
    rel = data["relationship_type"]
    w = RELATION_WEIGHTS.get(rel, 0)

    if G_simple.has_edge(u, v):
        G_simple[u][v]["weight"] += w
    else:
        G_simple.add_edge(u, v, weight=w)

print(f"Weighted simple graph: nodes={G_simple.number_of_nodes()}, edges={G_simple.number_of_edges()}")


# Triangles / Triads
triangles_dict = nx.triangles(G_simple)
triangles_count = sum(triangles_dict.values()) // 3

print("\n=== TRIADS / TRIANGLES ===")
print("Triangles (count):", triangles_count)
print("Global transitivity:", nx.transitivity(G_simple))


# Cliques
cliques = list(nx.find_cliques(G_simple))
large_cliques = [c for c in cliques if len(c) >= 4]

print("\n=== CLIQUES ===")
print("Total maximal cliques:", len(cliques))
print("Large cliques (>=4):", len(large_cliques))

# Count clique participation per node
clique_participation = {n: 0 for n in G_simple.nodes()}
for c in cliques:
    for n in c:
        clique_participation[n] += 1


# K-core
core_numbers = nx.core_number(G_simple)
print("Max k-core:", max(core_numbers.values()))


# Louvain community detection
print("\n=== COMMUNITY (Louvain) ===")
partition = community.best_partition(G_simple, weight="weight")
n_communities = len(set(partition.values()))

modularity = community.modularity(partition, G_simple, weight="weight")
print("Number of communities:", n_communities)
print("Modularity:", modularity)

# Group nodes by community

communities = defaultdict(list)
for node, com in partition.items():
    communities[com].append(node)

# Community plot (enhanced layout)
plt.figure(figsize=(14, 12))

# Create graph with strong intra-community attraction
H = G_simple.copy()
INTRA_FORCE = 5.0

for com, nodes in communities.items():
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            u, v = nodes[i], nodes[j]
            if not H.has_edge(u, v):
                H.add_edge(u, v, weight=INTRA_FORCE)
            else:
                H[u][v]["weight"] += INTRA_FORCE

# Layout influenced by community structure
pos = nx.spring_layout(H, seed=42, k=0.35, weight="weight")

# Assign colors to communities
sorted_coms = sorted(communities.keys(), key=lambda c: -len(communities[c]))
cmap = plt.cm.get_cmap("cool", n_communities)
community_color = {com: cmap(i) for i, com in enumerate(sorted_coms)}
node_colors = [community_color[partition[n]] for n in G_simple.nodes()]

# Node size: based on clique participation + core number
node_sizes = [
    50 + 120 * math.log(1 + clique_participation[n]) + 30 * core_numbers[n]
    for n in G_simple.nodes()
]

# Edge width proportional to weight
edge_weights = [G_simple[u][v]["weight"] for u, v in G_simple.edges()]
ew_max = max(edge_weights)
edge_widths = [0.5 + 3.5 * (w / ew_max) for w in edge_weights]

# Draw graph
nx.draw_networkx_edges(G_simple, pos, alpha=0.55, width=edge_widths)
nx.draw_networkx_nodes(
    G_simple, pos,
    node_size=node_sizes,
    node_color=node_colors,
    edgecolors="black", linewidths=0.6
)
nx.draw_networkx_labels(G_simple, pos, font_size=8)

plt.title(f"Louvain communities — {n_communities} groups — modularity {modularity:.3f}")
plt.axis("off")
plt.tight_layout()
plt.show()

# Clique plot
plt.figure(figsize=(12, 10))
pos = nx.spring_layout(G_simple, seed=42)

nx.draw_networkx_nodes(G_simple, pos, node_color="lightgray", node_size=300, edgecolors="black")
nx.draw_networkx_edges(G_simple, pos, alpha=0.3)
nx.draw_networkx_labels(G_simple, pos, font_size=8)

# Highlight large cliques
for clique in large_cliques:
    nx.draw_networkx_nodes(
        G_simple, pos,
        nodelist=clique,
        node_color="cyan",
        node_size=800,
        edgecolors="black"
    )

plt.title(f"Large cliques (≥4) — count: {len(large_cliques)}")
plt.axis("off")
plt.show()

# Community summary (table output)
print("\nCommunity summary:")
for com, nodes in sorted(communities.items(), key=lambda x: -len(x[1])):
    subg = G_simple.subgraph(nodes)
    dens = nx.density(subg)
    tri = sum(nx.triangles(subg).values()) // 3
    print(f" - community {com}: nodes={len(nodes)}, density={dens:.3f}, triangles={tri}")

print("\nPipeline completed.")
